In [1]:

import pandas as pd
import random
import re
import nltk

In [2]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [4]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [6]:
sports = [
("India Wins Cricket Series","India defeated Australia by six wickets in the final ODI after excellent batting and bowling."),
("Football Team Wins Championship","The football club scored two late goals to secure the league title."),
("Olympic Gold Medal","The athlete won a gold medal after breaking the national record."),
("Tennis Champion","The player lifted the Grand Slam trophy after an exciting final."),
("Kabaddi Tournament","The home team dominated the national kabaddi championship.")
]

politics = [
("New Tax Policy","The government announced a new tax policy to improve economic growth."),
("Election Campaign","Political leaders addressed thousands of supporters during the campaign."),
("Parliament Session","Members debated the new education bill in parliament."),
("Budget Announcement","The finance minister presented the annual budget."),
("International Summit","World leaders discussed climate change and trade policies.")
]

technology = [
("AI Startup Launches Product","A startup introduced an AI assistant for education and healthcare."),
("New Smartphone Released","The company launched a smartphone with advanced AI features."),
("Cyber Security","Experts warned about increasing cyber attacks across industries."),
("Cloud Computing","Businesses are adopting cloud technology to improve efficiency."),
("Software Update","The latest software update improves performance and security.")
]

business = [
("Stock Market Rises","The stock market reached a record high after strong earnings."),
("Company Expansion","The retail company announced expansion into international markets."),
("Startup Funding","The startup raised millions from investors."),
("Bank Profit","The bank reported higher quarterly profits."),
("Electric Vehicle Industry","Automobile companies increased investments in electric vehicles.")
]

entertainment = [
("Movie Breaks Records","The latest movie collected huge revenue worldwide."),
("Music Awards","Popular singers won multiple awards at the annual ceremony."),
("New Web Series","The streaming platform released a successful web series."),
("Film Festival","International filmmakers showcased their latest productions."),
("Celebrity Interview","The actor discussed upcoming projects during an interview.")
]

categories = {
    "Sports": sports,
    "Politics": politics,
    "Technology": technology,
    "Business": business,
    "Entertainment": entertainment
}


In [7]:
rows = []

article_id = 1

In [8]:
for category, articles in categories.items():
    for i in range(200):      # 200 articles per category

        title, article = random.choice(articles)

        rows.append({
            "Article_ID": article_id,
            "Title": title,
            "Article": article,
            "Category": category
        })

        article_id += 1

In [9]:
df = pd.DataFrame(rows)

In [10]:
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (1000, 4)
   Article_ID                            Title  \
0           1               Kabaddi Tournament   
1           2               Olympic Gold Medal   
2           3  Football Team Wins Championship   
3           4               Kabaddi Tournament   
4           5               Olympic Gold Medal   

                                             Article Category  
0  The home team dominated the national kabaddi c...   Sports  
1  The athlete won a gold medal after breaking th...   Sports  
2  The football club scored two late goals to sec...   Sports  
3  The home team dominated the national kabaddi c...   Sports  
4  The athlete won a gold medal after breaking th...   Sports  


In [11]:
df["Text"] = df["Title"] + " " + df["Article"]

In [12]:

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


In [13]:
def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z ]',' ',text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [14]:
df["Clean_Text"] = df["Text"].apply(preprocess)

In [17]:
X = df["Clean_Text"]
y = df["Category"]


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

In [21]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('classifier', MultinomialNB())])

In [23]:

pred = model.predict(X_test)

In [24]:
print("\nAccuracy:", accuracy_score(y_test, pred))

print("\nClassification Report\n")
print(classification_report(y_test, pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, pred))


Accuracy: 1.0

Classification Report

               precision    recall  f1-score   support

     Business       1.00      1.00      1.00        40
Entertainment       1.00      1.00      1.00        40
     Politics       1.00      1.00      1.00        40
       Sports       1.00      1.00      1.00        40
   Technology       1.00      1.00      1.00        40

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200


Confusion Matrix

[[40  0  0  0  0]
 [ 0 40  0  0  0]
 [ 0  0 40  0  0]
 [ 0  0  0 40  0]
 [ 0  0  0  0 40]]
